In [1]:
import sys
from pathlib import Path

# Get project root - the project root should have both 'vault' and 'ensemble' directories
# Walk up from current directory until we find a directory with both
current_dir = Path.cwd()
project_root = current_dir

# Walk up the directory tree to find project root
while project_root != project_root.parent:
    # Check if this directory has both vault and ensemble (indicating project root)
    has_vault = (project_root / 'vault').exists()
    has_ensemble = (project_root / 'ensemble').exists()
    
    if has_vault and has_ensemble:
        # Found the project root
        break
    
    project_root = project_root.parent
else:
    # If we reached the filesystem root without finding it, use parent of current_dir
    # (assuming we're in research/ subdirectory)
    if (current_dir.parent / 'vault').exists() and (current_dir.parent / 'ensemble').exists():
        project_root = current_dir.parent
    else:
        # Last resort: use current directory
        project_root = current_dir

# Add project root to Python path
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print(f"Current directory: {current_dir}")
print(f"Project root: {project_root}")
print(f"Has vault: {(project_root / 'vault').exists()}")
print(f"Has ensemble: {(project_root / 'ensemble').exists()}")
print(f"Python path includes: {str(project_root) in sys.path}")

Current directory: /home/raman/repos/Trading-Algo/research
Project root: /home/raman/repos/Trading-Algo
Has vault: True
Has ensemble: True
Python path includes: True


In [2]:
"""
Portfolio Testing with PortfolioTester

This notebook demonstrates how to use the PortfolioTester framework to:
1. Load data and create portfolios
2. Fit portfolios using candles
3. Generate predictions at different granularity levels
4. Calculate returns and generate tearsheets
5. Analyze performance at portfolio, ensemble, and base model levels
"""

from datetime import datetime
import pandas as pd
import numpy as np

from utils.enums import TimeFrame, Ticker
from utils import helpers
from ensemble.portfolio import Portfolio
from ensemble.diversified_ensemble import DiversifiedEnsemble
from ensemble.portfolio_tester import PortfolioTester

# Portfolio Testing Framework Demo

## Overview

The `PortfolioTester` class provides a comprehensive framework for testing portfolios:
- **Automatic returns calculation**: Calculates log returns from candles automatically
- **Granular predictions**: Get predictions at portfolio, ensemble, and base model levels
- **Integrated tearsheets**: Generate QuantStats tearsheets at all granularity levels
- **Baseline comparison**: Compare against equal-weight or buy-and-hold baselines

In [3]:
# Load data
print("Loading data...")



tickers = [ Ticker.ES, Ticker.NQ, Ticker.YM, Ticker.RTY]


tickers = [
    Ticker.ES,  # E-Mini S&P
    Ticker.NQ,  # E-Mini Nasdaq
    Ticker.YM,  # E-Mini Dow
    Ticker.RTY,  # E-Mini Russell 2000

    # Energy
    Ticker.CL,  # Crude Oil (WTI)
    Ticker.HO,  # ULSD (Heating Oil)

    # Metals
    Ticker.GC,  # Gold
    Ticker.HG,  # Copper
    Ticker.SI,  # Silver
    Ticker.PL,  # Platinum

    # Currencies (FX)
    Ticker.EU,  # Euro Currency
    Ticker.JY,  # Japanese Yen
    Ticker.BP,  # British Pound
    Ticker.CD,  # Canadian Dollar
    Ticker.SF,  # Swiss Franc

    # Agricultural (Food Grains)
    Ticker.C,   # Corn
    Ticker.S,   # Soybean
    Ticker.W,   # Wheat (SRW)

    # Meat
    Ticker.GF,  # Feeder Cattle

    # Fixed Income
    Ticker.TY,  # 10Yr T.Note
    Ticker.FV,  # 5Yr T.Note
    Ticker.US,  # 30Yr T.Bond
    Ticker.TU,  # 2Yr T.Note
]





train_candles = helpers.load_data_multi_ticker(
    tickers=tickers,
    timeframe=TimeFrame.D,
    start=datetime(2005, 1, 1),
    end=datetime(2020, 1, 1),
    use_millisecond_offset=True
)

test_candles = helpers.load_data_multi_ticker(
    tickers=tickers,
    timeframe=TimeFrame.D,
    start=datetime(2020, 1, 1),
    end=datetime(2024, 1, 1),
    use_millisecond_offset=True
)

print(f"Training data: {len(train_candles)} candles")
print(f"Test data: {len(test_candles)} candles")
print(f"Tickers: {sorted(train_candles['ticker'].unique())}")

Loading data...
Training data: 83912 candles
Test data: 23178 candles
Tickers: [<Ticker.ES: 'US500'>, <Ticker.NQ: 'US100'>, <Ticker.YM: 'US30'>, <Ticker.RTY: 'RUSSELL_2000'>, <Ticker.CL: 'OIL_CRUDE'>, <Ticker.HO: 'HEATING_OIL'>, <Ticker.GC: 'GOLD'>, <Ticker.HG: 'COPPER'>, <Ticker.SI: 'SILVER'>, <Ticker.PL: 'PLATINUM'>, <Ticker.EU: 'EURUSD'>, <Ticker.JY: 'YEN'>, <Ticker.BP: 'GBPUSD'>, <Ticker.CD: 'CADUSD'>, <Ticker.SF: 'CHFUSD'>, <Ticker.C: 'CORN'>, <Ticker.S: 'SOYBEANS'>, <Ticker.W: 'WHEAT'>, <Ticker.GF: 'FEEDER_CATTLE'>, <Ticker.TY: 'TEN_YR_NOTE'>, <Ticker.FV: 'FIVE_YR_NOTE'>, <Ticker.US: 'US_BONDS'>, <Ticker.TU: 'TWO_YR_NOTE'>]


## Step 1: Create Portfolio

For this demo, we'll use a simple buy-and-hold ensemble. In practice, you would load your ensemble from the vault or create it from feature configs.

In [4]:
# Load ensemble from vault using vault_manager
from ensemble.vault_manager import load_ensemble_from_vault

# Load ensembles from vault
USE_CACHE = True
TARGET_VOL = 0.15

ensemble_dirs = {
    # 'buy_hold': 'vault/D/buy-hold_indices_long',
    # 'mean_reversion_long': 'vault/D/mean-reversion_indices_long',
    # 'mean_reversion_short': 'vault/D/mean-reversion_indices_short',
    # 'momentum_long': 'vault/D/momentum_indices_long'
    'momo_long': 'vault/D/momo_long_long',
    'momo_short': 'vault/D/momo_short_short',
}


def _enable_cache(loaded_ensemble: DiversifiedEnsemble) -> DiversifiedEnsemble:
    loaded_ensemble.use_cache = USE_CACHE
    _ = [setattr(model, 'use_cache', USE_CACHE) for model in loaded_ensemble.base_models.values()]
    return loaded_ensemble


ensembles = {
    name: _enable_cache(
        load_ensemble_from_vault(path, refit=True, target_volatility=TARGET_VOL)
    )
    for name, path in ensemble_dirs.items()
}

print("Loaded ensembles:")
_ = [print(f"  {name}: {ensemble_dir}") for name, ensemble_dir in ensemble_dirs.items()]

# Create portfolio
portfolio = Portfolio(
    ensembles=[
        # ensembles['buy_hold'],
        # ensembles['mean_reversion_long'],
        # ensembles['mean_reversion_short'],
        # ensembles['momentum_long']
        ensembles['momo_long'],
        ensembles['momo_short'],
    ],
    trading_timeframe=TimeFrame.D,
    target_volatility=TARGET_VOL,
    max_position_pct=3.5,
    use_cache=USE_CACHE
)
print(f"\nPortfolio created with {len(portfolio.ensembles)} ensemble(s)")

Loaded ensembles:
  momo_long: vault/D/momo_long_long
  momo_short: vault/D/momo_short_short

Portfolio created with 2 ensemble(s)


## Step 2: Initialize PortfolioTester

The PortfolioTester wraps your portfolio and provides a convenient API for testing.

In [5]:
# Initialize PortfolioTester
tester = PortfolioTester(
    portfolio=portfolio,
    baseline_mode='equal_weight'  # or 'buy_hold'
)

# Fit portfolio (automatically calculates log returns)
print("\nFitting portfolio...")
tester.fit(train_candles)

# Print comprehensive diagnostics
tester.portfolio.print_diagnostics()

# Also print individual values for easy access
diagnostics = tester.portfolio.get_diagnostics()
print(f"\n📋 Quick Summary:")
print(f"  IDM: {diagnostics.get('idm', 'Not calculated')}")
print(f"  FDM: {diagnostics.get('fdm', 'Not calculated')}")
print(f"  Mean Return Correlation: {diagnostics.get('mean_return_correlation', 'N/A')}")
print(f"  Mean Forecast Correlation: {diagnostics.get('mean_forecast_correlation', 'N/A')}")



Fitting portfolio...


Partial cache coverage for ewmac (RTY, D). Cache starts at 2017-06-02 00:00:00, requested 2005-01-03 00:00:00. Proceeding with available data.



Fitting WeightLayer...


/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1335: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  candles_datetime_index['datetime'] = candles_datetime_index['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1337: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ensemble_result['datetime'] = ensemble_result['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1357: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  base_model_df['datetime'] = base_model_df['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1335: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  candles_datetime_index['datetime'] = candles_datetime_index['datetime'].dt.floor('S')
/home/ra


✓ WeightLayer fitted successfully!
  Tickers: 23
  Mean FDM: 1.0000 (range: 1.0000 - 1.0000)
  Mean models per ticker: 1.0
  Per-ticker FDM:
    Ticker.ES: FDM=1.0000 (1 model(s))
    Ticker.NQ: FDM=1.0000 (1 model(s))
    Ticker.YM: FDM=1.0000 (1 model(s))
    Ticker.RTY: FDM=1.0000 (1 model(s))
    Ticker.CL: FDM=1.0000 (1 model(s))
    Ticker.HO: FDM=1.0000 (1 model(s))
    Ticker.GC: FDM=1.0000 (1 model(s))
    Ticker.HG: FDM=1.0000 (1 model(s))
    Ticker.SI: FDM=1.0000 (1 model(s))
    Ticker.PL: FDM=1.0000 (1 model(s))
    Ticker.EU: FDM=1.0000 (1 model(s))
    Ticker.JY: FDM=1.0000 (1 model(s))
    Ticker.BP: FDM=1.0000 (1 model(s))
    Ticker.CD: FDM=1.0000 (1 model(s))
    Ticker.SF: FDM=1.0000 (1 model(s))
    Ticker.C: FDM=1.0000 (1 model(s))
    Ticker.S: FDM=1.0000 (1 model(s))
    Ticker.W: FDM=1.0000 (1 model(s))
    Ticker.GF: FDM=1.0000 (1 model(s))
    Ticker.TY: FDM=1.0000 (1 model(s))
    Ticker.FV: FDM=1.0000 (1 model(s))
    Ticker.US: FDM=1.0000 (1 model(s))
  

## Step 3: Generate Predictions

You can get predictions at different granularity levels:
- **Portfolio level**: Aggregated positions (default)
- **Ensemble level**: Individual ensemble predictions
- **Base model level**: Individual base model predictions

In [6]:
# Basic prediction (portfolio level only)
print("\nGenerating portfolio-level predictions...")
positions = tester.predict(test_candles)
print(f"Generated {len(positions)} position predictions")
print(f"Position fraction range: [{positions['position_fraction'].min():.4f}, {positions['position_fraction'].max():.4f}]")



Generating portfolio-level predictions...


/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1335: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  candles_datetime_index['datetime'] = candles_datetime_index['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1337: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ensemble_result['datetime'] = ensemble_result['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1357: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  base_model_df['datetime'] = base_model_df['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1335: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  candles_datetime_index['datetime'] = candles_datetime_index['datetime'].dt.floor('S')
/home/ra

Generated 23178 position predictions
Position fraction range: [0.0000, 0.1948]


/home/raman/repos/Trading-Algo/ensemble/portfolio.py:1486: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  forecasts_clean['datetime'] = forecasts_clean['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/portfolio.py:1492: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  candles_subset['datetime'] = candles_subset['datetime'].dt.floor('S')


In [7]:

# With ensemble-level and base model-level granularity
print("\nGenerating granular predictions (ensemble + base model level)...")
results = tester.predict(
    test_candles,
    return_ensemble_predictions=True,
    return_base_model_predictions=True
)

portfolio_positions = results['portfolio']
ensemble_predictions = results['ensembles']
base_model_predictions = results['base_models']

print(f"\nGranular predictions:")
print(f"  Portfolio positions: {len(portfolio_positions)}")
print(f"  Ensemble predictions: {len(ensemble_predictions)}")
print(f"  Base model predictions: {len(base_model_predictions)}")


Generating granular predictions (ensemble + base model level)...


/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1335: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  candles_datetime_index['datetime'] = candles_datetime_index['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1337: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ensemble_result['datetime'] = ensemble_result['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1357: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  base_model_df['datetime'] = base_model_df['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1335: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  candles_datetime_index['datetime'] = candles_datetime_index['datetime'].dt.floor('S')
/home/ra


Granular predictions:
  Portfolio positions: 23178
  Ensemble predictions: 2
  Base model predictions: 2


/home/raman/repos/Trading-Algo/ensemble/portfolio.py:1486: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  forecasts_clean['datetime'] = forecasts_clean['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/portfolio.py:1492: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  candles_subset['datetime'] = candles_subset['datetime'].dt.floor('S')


In [8]:
portfolio_positions

,ticker,datetime,forecast_score,position_fraction
0,Ticker.ES,2020-01-02,2.0,0.194786
1,Ticker.NQ,2020-01-02,0.0,0.000000
2,Ticker.YM,2020-01-02,0.0,0.000000
3,Ticker.RTY,2020-01-02,0.0,0.000000
4,Ticker.CL,2020-01-02,0.0,0.000000
...,...,...,...,...
23173,Ticker.GF,2023-12-29,2.0,0.194786
23174,Ticker.TY,2023-12-29,2.0,0.194786
23175,Ticker.FV,2023-12-29,2.0,0.194786
23176,Ticker.US,2023-12-29,2.0,0.194786


## Step 4: Calculate Returns

Calculate strategy returns and baseline returns for comparison.

In [9]:
# Calculate strategy returns
print("\nCalculating strategy returns...")
strategy_returns = tester.calculate_strategy_returns(test_candles)
print(f"Strategy returns: {len(strategy_returns)} days")
print(f"Total return: {strategy_returns.sum():.4f}")
if strategy_returns.std() > 0:
    sharpe = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)
    print(f"Annualized Sharpe: {sharpe:.4f}")

# Calculate baseline returns
print("\nCalculating baseline returns...")
baseline_returns = tester.calculate_baseline_returns(test_candles)
print(f"Baseline returns: {len(baseline_returns)} days")
print(f"Total return: {baseline_returns.sum():.4f}")
if baseline_returns.std() > 0:
    baseline_sharpe = baseline_returns.mean() / baseline_returns.std() * np.sqrt(252)
    print(f"Annualized Sharpe: {baseline_sharpe:.4f}")

# Compare performance
print(f"\nPerformance Comparison:")
print(f"  Strategy total return: {strategy_returns.sum():.4f} ({strategy_returns.sum()*100:.2f}%)")
print(f"  Baseline total return: {baseline_returns.sum():.4f} ({baseline_returns.sum()*100:.2f}%)")
print(f"  Excess return: {(strategy_returns.sum() - baseline_returns.sum()):.4f} ({(strategy_returns.sum() - baseline_returns.sum())*100:.2f}%)")


Calculating strategy returns...
Strategy returns: 1007 days
Total return: 0.4399
Annualized Sharpe: 0.3505

Calculating baseline returns...
Baseline returns: 23155 days
Total return: 3.2266
Annualized Sharpe: 0.1339

Performance Comparison:
  Strategy total return: 0.4399 (43.99%)
  Baseline total return: 3.2266 (322.66%)
  Excess return: -2.7866 (-278.66%)


/home/raman/repos/Trading-Algo/ensemble/portfolio_tester.py:104: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  candles_sorted['datetime'] = candles_sorted['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/portfolio_tester.py:121: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  positions['datetime'] = positions['datetime'].dt.floor('S')


## Step 5: Generate Tearsheets

Generate QuantStats tearsheets for performance analysis.

In [10]:
# Generate portfolio-level tearsheet
print("\nGenerating portfolio tearsheet...")
tester.generate_tearsheet(
    strategy_name='Buy Hold Portfolio',
    output_dir='reports/portfolio',  # Save HTML report to this directory
    mode='html',  # 'html', 'full', 'basic', or 'metrics'
    candles_df=test_candles
)

# Generate ensemble-level tearsheets (if available)
if tester.ensemble_predictions:
    print("\nGenerating ensemble tearsheets...")
    tester.generate_ensemble_tearsheets(
        output_dir='reports/ensembles',  # Save HTML reports to this directory
        mode='html',
        candles_df=test_candles
    )

# Generate base model-level tearsheets (if available)
if tester.base_model_predictions:
    print("\nGenerating base model tearsheets...")
    tester.generate_base_model_tearsheets(
        output_dir='reports/base_models',  # Save HTML reports to this directory
        mode='html',
        candles_df=test_candles
    )


Generating portfolio tearsheet...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


[OK] HTML tearsheet saved to: reports/portfolio/Buy_Hold_Portfolio_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)

Generating ensemble tearsheets...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


[OK] HTML tearsheet saved to: reports/ensembles/ensemble_0_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


...


[OK] HTML tearsheet saved to: reports/ensembles/ensemble_1_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)

Generating base model tearsheets...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


[OK] HTML tearsheet saved to: reports/base_models/ensemble_0_ewmac_signal_D_spanFast_[64]_spanSlow_[256]_two_bin_binning_2_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


[OK] HTML tearsheet saved to: reports/base_models/ensemble_1_ewmac_signal_D_spanFast_[64]_spanSlow_[256]_two_bin_binning_2_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


In [11]:
# Generate training-set tearsheets (predict on train period, then restore test state)
print("\nGenerating training-set tearsheets...")

from ensemble.portfolio_tester import calculate_strategy_returns_from_positions, calculate_baseline_returns

# Get positions for training period (overwrites tester state temporarily)
tester.predict(
    train_candles,
    return_ensemble_predictions=bool(tester.ensemble_predictions),
    return_base_model_predictions=bool(tester.base_model_predictions),
)

train_strategy_returns = calculate_strategy_returns_from_positions(
    tester.positions_df, train_candles, strategy='long'
)
train_baseline_returns = calculate_baseline_returns(
    train_candles, equal_weight=(tester.baseline_mode == 'equal_weight')
)

tester.generate_tearsheet(
    strategy_name='Buy Hold Portfolio (Train)',
    output_dir='reports/portfolio_train',
    mode='html',
    strategy_returns=train_strategy_returns,
    baseline_returns=train_baseline_returns,
)

if tester.ensemble_predictions:
    print("\nGenerating training ensemble tearsheets...")
    tester.generate_ensemble_tearsheets(
        output_dir='reports/ensembles_train',
        mode='html',
        candles_df=train_candles,
        baseline_returns=train_baseline_returns,
    )

if tester.base_model_predictions:
    print("\nGenerating training base model tearsheets...")
    tester.generate_base_model_tearsheets(
        output_dir='reports/base_models_train',
        mode='html',
        candles_df=train_candles,
        baseline_returns=train_baseline_returns,
    )

# Restore test-set predictions for any later cells
tester.predict(
    test_candles,
    return_ensemble_predictions=bool(tester.ensemble_predictions),
    return_base_model_predictions=bool(tester.base_model_predictions),
)


Generating training-set tearsheets...


/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1335: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  candles_datetime_index['datetime'] = candles_datetime_index['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1337: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ensemble_result['datetime'] = ensemble_result['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1357: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  base_model_df['datetime'] = base_model_df['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1335: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  candles_datetime_index['datetime'] = candles_datetime_index['datetime'].dt.floor('S')
/home/ra


[OK] HTML tearsheet saved to: reports/portfolio_train/Buy_Hold_Portfolio_(Train)_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)

Generating training ensemble tearsheets...


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


[OK] HTML tearsheet saved to: reports/ensembles_train/ensemble_0_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


...


[OK] HTML tearsheet saved to: reports/base_models_train/ensemble_0_ewmac_signal_D_spanFast_[64]_spanSlow_[256]_two_bin_binning_2_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font family 'Arial' not found.
findfont: Font f


[OK] HTML tearsheet saved to: reports/base_models_train/ensemble_1_ewmac_signal_D_spanFast_[64]_spanSlow_[256]_two_bin_binning_2_tearsheet.html
   Note: Using non-compounded returns (Robert Carver methodology)


/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1335: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  candles_datetime_index['datetime'] = candles_datetime_index['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1337: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  ensemble_result['datetime'] = ensemble_result['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1357: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  base_model_df['datetime'] = base_model_df['datetime'].dt.floor('S')
/home/raman/repos/Trading-Algo/ensemble/diversified_ensemble.py:1335: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  candles_datetime_index['datetime'] = candles_datetime_index['datetime'].dt.floor('S')
/home/ra

{'portfolio':            ticker   datetime  forecast_score  position_fraction
 0       Ticker.ES 2020-01-02             2.0           0.194786
 1       Ticker.NQ 2020-01-02             0.0           0.000000
 2       Ticker.YM 2020-01-02             0.0           0.000000
 3      Ticker.RTY 2020-01-02             0.0           0.000000
 4       Ticker.CL 2020-01-02             0.0           0.000000
 ...           ...        ...             ...                ...
 23173   Ticker.GF 2023-12-29             2.0           0.194786
 23174   Ticker.TY 2023-12-29             2.0           0.194786
 23175   Ticker.FV 2023-12-29             2.0           0.194786
 23176   Ticker.US 2023-12-29             2.0           0.194786
 23177   Ticker.TU 2023-12-29             2.0           0.194786
 
 [23178 rows x 4 columns],
 'ensembles': {'ensemble_0':            ticker   datetime  forecast_score  position_fraction
  0       Ticker.ES 2020-01-02             2.0           0.194786
  1       Ticker.NQ